# Week 10 Lab — Collaborative Filtering with ALS

**Goal:** Train an Alternating Least Squares recommender on MovieLens 1M. Build the model, evaluate it, generate predictions, and experiment with parameters.

You've already done EDA. Now apply what you learned to build a real recommender.

## Setup

In [1]:
from pyspark.sql import SparkSession
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.sql.functions import col, explode
import math

spark = SparkSession.builder.appName("ALS_Lab").getOrCreate()


## Load Data

In [2]:
import os
import urllib.request
import zipfile

# Your data path — set once, reuse for all runs
data_folder = os.path.abspath("../ml-1m")  # Use absolute path for reliability

ratings_path = os.path.join(data_folder, "ratings.dat")
movies_path = os.path.join(data_folder, "movies.dat")

# Check if data exists; if not, download and extract MovieLens 1M
if not os.path.exists(ratings_path):
    print("MovieLens 1M data not found. Downloading...")
    os.makedirs(data_folder, exist_ok=True)
    url = "https://files.grouplens.org/datasets/movielens/ml-1m.zip"
    zip_path = os.path.join(data_folder, "ml-1m.zip")
    urllib.request.urlretrieve(url, zip_path)
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(os.path.dirname(data_folder))
    os.remove(zip_path)
    print("Download and extraction complete.")

# Load ratings (userId::movieId::rating::timestamp)
# ALS requires numeric columns — cast all IDs and rating to the right types
ratings = spark.read.option("sep", "::").option("header", False) \
    .csv(ratings_path).toDF("userId", "movieId", "rating", "timestamp") \
    .select(
        col("userId").cast("int"),
        col("movieId").cast("int"),
        col("rating").cast("float"),
        col("timestamp")
    )

# Load movies (movieId::title::genres)
# Cast movieId to int so joins with ratings work correctly
movies = spark.read.option("sep", "::").option("header", False) \
    .csv(movies_path).toDF("movieId", "title", "genres") \
    .select(
        col("movieId").cast("int"),
        col("title"),
        col("genres")
    )

print(f"{ratings.count()} ratings loaded")
print(f"{movies.count()} movies loaded")
ratings.printSchema()


1000209 ratings loaded
3883 movies loaded
root
 |-- userId: integer (nullable = true)
 |-- movieId: integer (nullable = true)
 |-- rating: float (nullable = true)
 |-- timestamp: string (nullable = true)



## Train/Test Split

80% train, 20% test. You've done this before — give it a try.

In [3]:
# YOUR CODE HERE:
# Use: randomSplit() with weights [0.8, 0.2] and seed=42
# Hint: ratings.randomSplit(...)

train, test = ratings.randomSplit([0.8, 0.2], seed=42)

# Validation: Check the split
train_count = train.count()
test_count = test.count()
total_count = train_count + test_count
train_pct = train_count / total_count * 100

print(f"Train: {train_count} | Test: {test_count}")
print(f"Train ratio: {train_pct:.1f}% (should be ~80%)")
assert 79 < train_pct < 81, f"Train ratio {train_pct:.1f}% is too far from 80%"
print("✓ Split is correct")

Train: 800092 | Test: 200117
Train ratio: 80.0% (should be ~80%)
✓ Split is correct


## Build and Train ALS

Create and train an ALS recommender.

**Hint:** Use the `ALS` class. You'll need to set:
- `rank` — number of **latent dimensions (D)** to learn for each user/movie
- `maxIter` (start with 10) — iterations until convergence
- `regParam` (start with 0.01) — regularization strength
- `userCol`, `itemCol`, `ratingCol` (the column names)
- `seed` (for reproducibility)

Then call `.fit(train)` to train. Start with rank=10 (so D=10).

In [4]:
# YOUR CODE HERE:
# Create an ALS object and train it on the training set

als = ALS(
    rank=10,
    maxIter=10,
    regParam=0.01,
    userCol="userId", itemCol="movieId", ratingCol="rating",
    coldStartStrategy="drop",   # drop NaN predictions from cold-start users/movies
    seed=42
)
model = als.fit(train)

# Validation: Check model exists and has expected attributes
assert model is not None, "Model should be trained"
assert hasattr(model, 'transform'), "Model should have transform method"
print(f"✓ Model trained successfully")
print(f"  rank={als.getRank()}, maxIter={als.getMaxIter()}, regParam={als.getRegParam()}")


✓ Model trained successfully
  rank=10, maxIter=10, regParam=0.01


## Evaluate: RMSE

Compute how well the model predicts ratings on the test set.

**Hint:** Use `RegressionEvaluator` with parameters:
- `metricName="rmse"`
- `labelCol="rating"`
- `predictionCol="prediction"`

First generate predictions with `model.transform(test)`.

In [5]:
# 1. Generate predictions on the test set
predictions = model.transform(test)

# 2. Create a RegressionEvaluator and compute RMSE
evaluator = RegressionEvaluator(metricName="rmse", labelCol="rating", predictionCol="prediction")
rmse = evaluator.evaluate(predictions)

# Validation: Check RMSE is reasonable
assert rmse is not None, "RMSE should be computed"
assert 0.5 < rmse < 2.0, f"RMSE {rmse:.4f} seems unreasonable (should be ~0.85–0.95)"
print(f"✓ Test RMSE: {rmse:.4f}")
print("  (Lower is better. On 1–5 star scale, RMSE of 0.9 means ±0.9 star error on average)")


✓ Test RMSE: 0.8888
  (Lower is better. On 1–5 star scale, RMSE of 0.9 means ±0.9 star error on average)


## Generate Recommendations

Pick a user and see what movies ALS recommends for them.

**Hint:** Use `model.recommendForUserSubset()` to get top-K recommendations. You'll need to:
1. Create a DataFrame with the user ID
2. Call the method with the DataFrame and number of recommendations
3. Flatten the recommendations array using `explode()`
4. Join with the movies DataFrame to get titles

In [ ]:
# YOUR CODE HERE:
# Generate top 5 recommendations for user 100
# Then flatten and join with movie titles

user_id = 100

user_recs = None
recs_flat = None
recs_with_titles = None

# Validation: Check we got recommendations
assert recs_with_titles is not None, "Should have recommendations"
rec_count = recs_with_titles.count()
assert rec_count > 0, f"Should have at least 1 recommendation, got {rec_count}"
print(f"\n✓ Got {rec_count} recommendations for user {user_id}:")
recs_with_titles.select("title", "predicted_rating").show(truncate=False)

## Evaluate: Precision@K and NDCG

RMSE measures prediction accuracy across all ratings. But users only see top-K recommendations. Let's measure ranking quality.

**Precision@K:** Of the top K recommendations, how many did the user actually like (rated ≥ 4)?

**NDCG (Normalized Discounted Cumulative Gain):** Measures ranking quality — puts more weight on relevant items ranked higher.

**Hint:** For a user in the test set:
1. Get their top 10 recommendations
2. Find items they rated ≥ 4 in the test set ("liked")
3. Check which recommendations overlap with liked items
4. Compute Precision@10 = (# liked in top 10) / 10
5. Compute NDCG@10 using the formula from the whiteboard guide

In [ ]:
# Helper function to compute NDCG@K
def compute_ndcg(recommended_items, relevant_items, k=10):
    """
    Compute NDCG@K.
    - recommended_items: list of itemIds in recommendation order
    - relevant_items: set of itemIds the user liked (rated >= 4)
    - k: top-K cutoff
    """
    # Discounted Cumulative Gain: sum of (1 if relevant else 0) / log2(position + 1)
    dcg = 0.0
    for i, item_id in enumerate(recommended_items[:k]):
        if item_id in relevant_items:
            dcg += 1.0 / math.log2(i + 2)  # position is i+1, log2(i+2) = log2(position+1)
    
    # Ideal DCG: best case = all relevant items at top
    idcg = 0.0
    for i in range(min(len(relevant_items), k)):
        idcg += 1.0 / math.log2(i + 2)
    
    # NDCG = DCG / IDCG (0 if no relevant items)
    if idcg == 0:
        return 0.0
    return dcg / idcg

print("✓ NDCG function defined")

In [ ]:
# YOUR CODE HERE:
# Pick a few users and compute Precision@10 and NDCG@10
# Steps:
# 1. For each user: get their top 10 recommendations
# 2. Get items they rated >= 4 in the test set
# 3. Compute precision@10: (# liked items in top 10) / 10
# 4. Compute NDCG@10 using the function above
# 5. Print results

K = 10
test_users = [100, 200, 300]  # Sample a few users

print(f"\nPrecision@{K} and NDCG@{K} for sample users:")
print("-" * 60)
print(f"{'User':>6} | {'Precision@K':>12} | {'NDCG@K':>12}")
print("-" * 60)

# YOUR CODE HERE: Compute Precision@K and NDCG@K for each user
# Hints:
# - model.recommendForUserSubset() to get top K
# - test.filter(col("userId") == user_id) to get user's test ratings
# - test_ratings.filter(col("rating") >= 4) for "liked" items
# - Use compute_ndcg() function above

for user_id in test_users:
    precision_at_k = None  # YOUR CODE HERE
    ndcg_at_k = None  # YOUR CODE HERE
    
    if precision_at_k is not None and ndcg_at_k is not None:
        print(f"{user_id:>6} | {precision_at_k:>12.4f} | {ndcg_at_k:>12.4f}")

print("-" * 60)
print("→ Compare to RMSE: which metric matters more for real recommendations?")

## Experiment: Parameter Tuning

Try different `rank` values (D = dimensionality). Does RMSE improve? Is there a sweet spot where it stops improving?

From the lecture: R ≈ U × M^T where U is (users × **D**) and M is (movies × **D**).

**Note:** D (latent dimensions) ≠ K (top-K in Precision@K). Don't confuse them.

**Hint:** For each rank value, you'll repeat the same steps:
1. Create an ALS object with the new rank value
2. Train it on the training set
3. Generate predictions on the test set
4. Evaluate RMSE
5. Record the result

In [ ]:
# Experiment with rank values
rank_values = [5, 10, 20, 30, 50]  # Modify as you explore
results = []

for r in rank_values:
    # YOUR CODE HERE:
    # Create ALS with rank=r (keep maxIter=10, regParam=0.01)
    # Train on train set, evaluate on test set
    
    als_temp = None
    model_temp = None
    pred_temp = None
    rmse_temp = None
    
    results.append((r, rmse_temp))
    print(f"rank={r:2d} → RMSE={rmse_temp:.4f}")

# Validation: Check we got results for each rank
assert len(results) == len(rank_values), f"Should have {len(rank_values)} results, got {len(results)}"
print("\n✓ Parameter tuning complete")
print("→ What pattern do you see? Does higher rank always help?")